In [1]:
import os
import time
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor
from tqdm import tqdm

import barostat_parameters
from barostat_utils import (
    estimate_initial_box_vel_y,
    estimate_initial_box_vel_y_accurate,
    update_box_y_thermodynamic,
)
from graph_utils import LJInteractionParams, prepare_traj
from itpo_weights import DatasetType
from pressure import compute_per_particle_forces
from simulator_SA_cpu_test import Model as VelocityModel
from training_utils import (
    ModelInputs,
    huber_loss,
)
from utils import (
    build_velocity_graph_correction,
    calc_p_ratio_box_tensor,
    get_correct_edge_attr,
    load_and_split_dataset,
    visualize_nu_disribution,
)


### Load Data

In [ ]:
# poisson_buckets = [
#     {"max": 0.1, "count": 300},               # P < 0.1
#     {"min": 0.1, "max": 0.2, "count": 100},   # 0.1 <= P < 0.2
#     {"min": 0.2, "count": 100}                # P >= 0.2
# ]

poisson_buckets = [
    {"min": 0.0, "max": 1.0, "count": 200},   # full range
]

dataset_type = DatasetType.LJNoisy

train_files, val_files, test_files = load_and_split_dataset(
    # registry_path="./data_mini/data_registry_mini.csv",
    # registry_path="./data/data_registry.csv",
    # registry_path="./data_reid_with_angles/data_registry.csv",
    registry_path="./data/data_LJ_noisy_eps0.01_sigma1.0_cutoff1.122/data_registry.csv",
    target_data_type=dataset_type,
    possion_buckets=poisson_buckets,
    split_ratios=(0.5, 0.25, 0.25),
    seed=42
)

# Load actual data
data = {
    'train': {},
    'val' : {},
    'test' : {},
}

max_sim_len = 200
print(f"Loading data with max simulation length of {max_sim_len}...")
# stride = 5
# print(f"Loading data with additional stride of {stride}.")
for key in data.keys():
    if key == 'train':
        data[key] = [torch.load(file, weights_only=False)[:max_sim_len] for file in tqdm(train_files, desc=f"{key:<5} data")]
        # data[key] = [torch.load(file, weights_only=False)[::stride] for file in tqdm(train_files, desc=f"{key:<5} data")]
    elif key == 'val':
        data[key] = [torch.load(file, weights_only=False)[:max_sim_len] for file in tqdm(val_files, desc=f"{key:<5} data")]
        # data[key] = [torch.load(file, weights_only=False)[::stride] for file in tqdm(val_files, desc=f"{key:<5} data")]
    elif key == 'test':
        data[key] = [torch.load(file, weights_only=False)[:max_sim_len] for file in tqdm(test_files, desc=f"{key:<5} data")]
        # data[key] = [torch.load(file, weights_only=False)[::stride] for file in tqdm(test_files, desc=f"{key:<5} data")]
    else:
        raise ValueError(f"Unexpected key in data dictionary: {key}. ")

print("\nPreparing data...")
for data_type, sims in data.items():
    prepared = []
    for sim in tqdm(sims, desc=f"{data_type:<5} data"):
        lj_params = LJInteractionParams(epsilon=0.01, sigma=1.0, cutoff=1.122)
        prepared_sim = prepare_traj(sim, lj_params, calc_angles=False)
        prepared.append(prepared_sim)
    data[data_type] = prepared

print(f"\nTrain data: {len(data['train'])} sims.")
print(f"Val data:   {len(data['val'])} sims.")
print(f"Test data:  {len(data['test'])} sims.")

visualize_nu_disribution(data)


### Training GNN simulator

#### Initialize Velocity simulator

In [ ]:
mp_layers = 2
mlp = 3
hidden_size = 128
history = 3
device = "cuda"

init_graph = build_velocity_graph_correction(
    input_graphs=[data['train'][0][i].cpu().detach() for i in range(history + 1)],
    total_velocity=False,
    panic_at_positions=False
).to(device)

gnn_simulator = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)


#### One-step Training loop

Here we train on a portion of training data with high Poisson's ratio, either $\nu > 0.1$ or $\nu > 0.2$.

In [ ]:
model_save_directory = os.path.join("./LJ_trained_models", f"{dataset_type}", "OST")
if not os.path.exists(model_save_directory):
    os.makedirs(model_save_directory, exist_ok=True)

epochs = 150
freeze_norm_epoch = 5
train_sims = 100
val_sims = 20
train_limit = 15
accumulation_steps = 10
learning_rate = 1e-3
gamma = 0.995

# Limit training data to high Poisson's ratio simulation
# poisson_threshold = 0.1
# training_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= poisson_threshold][:train_sims]
# print(f"Using {len(training_data)} simulations with Poisson's ratio >= {poisson_threshold} for training. ")

training_data = data['train'][:train_sims]
print(f"Using {len(training_data)} simulations with full range of Poisson's ratios for training. ")

# Validation data can be full Poisson's ratio range

optimizer = torch.optim.Adam(gnn_simulator.parameters(), lr=learning_rate, weight_decay=0.0)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)

optimizer.zero_grad()
for epoch in range(epochs):
    t_start = time.perf_counter()

    if epoch == freeze_norm_epoch:
        gnn_simulator.node_normalizer.frozen = True
        gnn_simulator.edge_normalizer.frozen = True
        gnn_simulator.output_normalizer.frozen = True

    total_acc_loss = 0
    total_val_loss = 0
    total_val_pos_mse = 0
    train_samples = 0
    val_samples = 0

    gnn_simulator.train()
    for sim in training_data:
        starting_points = [i for i in range(train_limit)]

        for i, idx in enumerate(starting_points):
            indices = [step + idx for step in range(history + 1)]
            target_idx = history + 1 + idx

            input_graphs_raw = [sim[k].detach().cpu() for k in indices]

            # Construct input graph
            input_graph = build_velocity_graph_correction(input_graphs_raw, panic_at_positions=False).to(device)

            # Construct ModelInputs
            model_inputs = ModelInputs(
                input_graphs_raw[-2].to(device) if history > 0 else input_graphs_raw[-1].to(device),
                input_graphs_raw[-1].to(device),
                sim[target_idx].to(device)
            )

            # Forward and Loss
            model_output = gnn_simulator(input_graph, is_training=True)
            acc_loss = huber_loss(gnn_simulator, model_output, model_inputs, is_training=True)

            # Backward
            loss_for_backward = acc_loss / accumulation_steps
            loss_for_backward.backward()

            # Optimization
            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(gnn_simulator.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            total_acc_loss += acc_loss.item()
            train_samples += 1

        optimizer.step()
        optimizer.zero_grad()

    # Validation
    with torch.no_grad():
        gnn_simulator.eval()
        for val_sim in data['val'][:val_sims]:

            starting_points = [i for i in range(train_limit)]

            for idx in starting_points:
                indices = [step + idx for step in range(history + 1)]
                target_idx = history + 1 + idx

                input_graphs_raw = [val_sim[k].detach().cpu() for k in indices]

                input_graph = build_velocity_graph_correction(input_graphs_raw, panic_at_positions=False).to(device) 

                val_inputs = ModelInputs(
                    input_graphs_raw[-2].to(device) if history > 0 else input_graphs_raw[-1].to(device),
                    input_graphs_raw[-1].to(device),
                    val_sim[target_idx].to(device),
                )

                # Forward and Loss
                model_output = gnn_simulator(input_graph, is_training=False)
                val_loss = huber_loss(gnn_simulator, model_output, val_inputs, is_training=False)

                # Update to next state and check position MSE
                pred_graph = gnn_simulator.update(val_inputs, model_output)
                pos_mse = torch.nn.functional.mse_loss(pred_graph.pos, val_sim[target_idx].to(device).pos)

                total_val_loss += val_loss.item()
                total_val_pos_mse += pos_mse.item()
                val_samples += 1

    lr_scheduler.step()

    # Statistics
    avg_train_loss = total_acc_loss / train_samples
    avg_val_loss = total_val_loss / val_samples
    avg_val_pos_mse = total_val_pos_mse / val_samples

    # Save model
    if epoch % 1 == 0:
        gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"checkpoint_epoch_{epoch}.pt"))

    t_stop = time.perf_counter()
    print(
        f"Epoch {epoch + 1:>3} | "
        f"Train Loss: {avg_train_loss:.3e} | "
        f"Val Loss: {avg_val_loss:.3e} | "
        f"Val Pos MSE: {avg_val_pos_mse:.3e} | "
        f"Time: {t_stop - t_start:.2f} s"
    )

gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"model_h{history}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))


#### Multi-step training loop

In [ ]:
def detach_to_cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu()
    return x

# model_save_directory = f"./full_range_models/{dataset_type}/MST"
# model_save_directory = f"./angles_test/{dataset_type}/MST"
model_save_directory = f"./LJ_trained_models/{dataset_type}/MST"
if not os.path.exists(model_save_directory):
    os.makedirs(model_save_directory, exist_ok=True)

epochs = 100
train_sims = 100
train_limit: int = 15
max_rollout_steps: int = 10
fresh: bool = True
freeze_norm_epoch: int = 5
learning_rate: float = 1e-3
gamma = 0.995

# Limit training data to high Poisson's ratio simulation
# poisson_threshold = 0.1
# training_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= poisson_threshold][:train_sims]
# print(f"Using {len(training_data)} simulations with Poisson's ratio >= {poisson_threshold} for training. ")

training_data = data['train'][:train_sims]
print(f"Using {len(training_data)} simulations with full range of Poisson's ratios for training. ")

if dataset_type is DatasetType.NodeOptimized:
    barostat_config = barostat_parameters.node_optimizated 
elif dataset_type is DatasetType.StiffOptimized:
    barostat_config = barostat_parameters.stiff_optimized
elif dataset_type is DatasetType.StiffAngles:
    barostat_config = barostat_parameters.stiff_angles
elif dataset_type is DatasetType.Noisy:
    barostat_config = barostat_parameters.noisy
elif dataset_type is DatasetType.LJNoisy:
    barostat_config = barostat_parameters.lj_noisy

if barostat_config["default_skip"] is None:
    print("Dump freqiency is computed dynamically.")

gnn_simulator.train()
params = filter(lambda p: p.requires_grad, gnn_simulator.parameters())
optimizer = torch.optim.Adam(params, lr=learning_rate, weight_decay=0.0)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)

optimizer.zero_grad()
for epoch in range(epochs):
    t_start = time.perf_counter()

    if fresh:
        if epoch < 10:
            rollout_steps = 1
        elif epoch >= 10 and epoch < 20:
            rollout_steps = 2
        elif epoch >= 20 and epoch < 30:
            rollout_steps = 3
        elif epoch >= 30 and epoch < 40:
            rollout_steps = 5
        elif epoch >= 40 and epoch < 50:
            rollout_steps = 8
        else:
            rollout_steps = max_rollout_steps

    # Trackers
    total_acc_loss = 0
    train_samples = 0

    # Freeze normalizers
    if epoch == freeze_norm_epoch:
        gnn_simulator.node_normalizer.frozen = True
        gnn_simulator.edge_normalizer.frozen = True
        gnn_simulator.output_normalizer.frozen = True

    for sim in training_data:
        # Get equilibrium bond lengths
        r0 = sim[0].edge_attr[:, -2]
        
        starting_points = [i for i in range(train_limit)]
        if barostat_config["default_skip"] is not None:
            dump_period = barostat_config["default_skip"]
        else:
            sim_strain = (sim[1].box.x - sim[-1].box.x) / sim[0].box.x
            assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
            dump_period = int(assumed_rollout_length / len(sim)) + 1

        for start_idx in starting_points:
            optimizer.zero_grad()

            indices = [step + start_idx for step in range(history + 1)]
            current_window_graphs = [sim[k].detach().to(device) for k in indices]

            b0 = current_window_graphs[-2].box_tensor[0]
            b1 = current_window_graphs[-1].box_tensor[0]
            box_delta_x = b1 - b0

            if len(current_window_graphs) < 3:
                current_box_vel_y = estimate_initial_box_vel_y(
                    current_window_graphs[-2],
                    current_window_graphs[-1],
                    dump_period * barostat_config["dt"],
                )
            elif len(current_window_graphs) >= 3:
                current_box_vel_y = estimate_initial_box_vel_y_accurate(
                    current_window_graphs[-3],
                    current_window_graphs[-2],
                    current_window_graphs[-1],
                    dump_period * barostat_config["dt"],
                )
            else:
                raise Exception(f"Window size is too small : {len(current_window_graphs)}")


            rollout_loss = 0
            for step in range(rollout_steps):
                target_idx = history + 1 + start_idx + step
                target_graph = sim[target_idx].to(device)

                input_graph = build_velocity_graph_correction(current_window_graphs).to(device)

                model_inputs = ModelInputs(
                    current_window_graphs[-2],
                    current_window_graphs[-1],
                    target_graph,
                )

                model_output = gnn_simulator(input_graph, is_training=True)
                pred_graph_next = gnn_simulator.update(model_inputs, model_output)

                step_loss = huber_loss(gnn_simulator, model_output, model_inputs, is_training=True)
                rollout_loss += step_loss

                dt = barostat_config["dt"]  # lammps dt
                W_y = barostat_config["C_coupling"] * pred_graph_next.num_nodes * ((dump_period * dt) ** 2)
                damping = barostat_config["damping"] * pred_graph_next.num_nodes * (dump_period * dt)

                new_lx = pred_graph_next.box_tensor[0] + box_delta_x
                new_ly, new_vel_y = update_box_y_thermodynamic(
                    positions=pred_graph_next.pos,
                    edge_index=model_inputs.cur_graph.edge_index,
                    edge_attr=model_inputs.cur_graph.edge_attr,
                    current_box=model_inputs.cur_graph.box_tensor,
                    r0=r0.to(pred_graph_next.pos.device),
                    box_vel_y=current_box_vel_y,  # Use ESTIMATED velocity
                    W_y=W_y,
                    damping=damping,
                    stride_dt=dump_period * dt,
                    lj_cutoff=lj_params.cutoff,
                    target_pressure=barostat_config["target_pressure"],
                    temperature=barostat_config["temperature"],
                )
                new_box_tensor = torch.stack([new_lx, new_ly])
                current_box_vel_y = new_vel_y

                # Add new box and update edge_attr
                pred_graph_next.box_tensor = new_box_tensor
                
                function_output = get_correct_edge_attr(
                    pred_graph_next,
                    recompute_stiff=False,
                    lj_params=lj_params,
                    panic_at_nontensor_box=True,
                )
                if isinstance(function_output, Tensor):
                    pred_graph_next.edge_attr = function_output
                
                elif isinstance(function_output, Tuple):
                    edge_index, edge_attr = function_output
                    pred_graph_next.edge_index = edge_index
                    pred_graph_next.edge_attr = edge_attr
                
                pred_graph_next.forces = compute_per_particle_forces(pred_graph_next, r0=r0.to(pred_graph_next.pos.device), cutoff=lj_params.cutoff)

                pred_graph_next_detached = pred_graph_next.detach()

                # Update window: shift left, append new prediction
                current_window_graphs.pop(0)
                current_window_graphs.append(pred_graph_next_detached)

            final_loss = rollout_loss / rollout_steps
            final_loss.backward()

            # Clip gradients
            torch.nn.utils.clip_grad_norm_(gnn_simulator.parameters(), max_norm=1.0)
            optimizer.step()

            total_acc_loss += final_loss.item()
            train_samples += 1

    gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"checkpoint_epoch_{epoch}.pt"))

    total_acc_loss /= train_samples
    lr_scheduler.step()
    t_stop = time.perf_counter()
    print(f"Epoch {epoch:<3} | steps: {rollout_steps:<2} | loss: {total_acc_loss:.4e} | {t_stop - t_start:.2f} s.")

gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"model_h{history}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))


In [ ]:
def get_sampled_files_with_labels(
    registry_path: str,
    target_data_type: DatasetType,
    poisson_buckets: list,
    seed: int = 42
) -> tuple[list[str], list[int]]:
    """
    Samples data files and returns a tuple of:
    1. A shuffled list of file paths.
    2. A matching list of bucket integer IDs to use for stratification.
    """
    df = pd.read_csv(registry_path)
    type_df = df[df['data_type'] == str(target_data_type)]
    
    if type_df.empty:
        raise ValueError(f"No data found for data_type: {str(target_data_type)}")

    sampled_dfs = []

    for i, bucket in enumerate(poisson_buckets):
        p_min = bucket.get('min', float('-inf'))
        p_max = bucket.get('max', float('inf'))
        req_count = bucket['count']

        bucket_df = type_df[(type_df['poisson_ratio'] >= p_min) & (type_df['poisson_ratio'] < p_max)].copy()
        
        available = len(bucket_df)
        if available == 0:
            logger.warning(f"Bucket {i} ({p_min} <= P < {p_max}) is empty.")
            continue
            
        if req_count > available:
            req_count = available
            
        sampled = bucket_df.sample(n=req_count, random_state=seed)
        
        # Tag each row with its specific bucket index for stratification
        sampled['bucket_id'] = i  
        sampled_dfs.append(sampled)

    if not sampled_dfs:
        raise ValueError("No data was sampled from any bucket. Check threshold logic.")

    # Combine and shuffle while maintaining the index match between files and bucket_ids
    combined_df = pd.concat(sampled_dfs).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return combined_df['file_path'].tolist(), combined_df['bucket_id'].tolist()


def train_model_once(data: dict[str, List], model: VelocityModel, epochs: int, barostat_config: Dict, save_dir: str) -> Dict:
    
    freeze_norm_epoch = 5
    train_sims = 100
    val_sims = 100
    train_limit = 15
    accumulation_steps = 10
    learning_rate = 1e-3
    gamma = 0.995

    num_rollout_steps = 50
    history = 3

    # Limit training data
    poisson_threshold = 0.1
    training_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= poisson_threshold][:train_sims]
    logger.info(f"Using {len(training_data)} simulations with Poisson's ratio >= {poisson_threshold} for training.")
    
    # Changed `gnn_simulator` to `model` to match function arguments
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.0)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)

    results = {
        "training_loss": [], 
        "r2": [], 
        "rollout_mse": [] 
    }

    optimizer.zero_grad()
    for epoch in range(epochs):
        t_start = time.perf_counter()

        if epoch == freeze_norm_epoch:
            model.node_normalizer.frozen = True
            model.edge_normalizer.frozen = True
            model.output_normalizer.frozen = True

        total_acc_loss = 0
        train_samples = 0

        model.train()
        for sim in training_data:
            starting_points = [i for i in range(train_limit)]

            for i, idx in enumerate(starting_points):
                indices = [step + idx for step in range(history + 1)]
                target_idx = history + 1 + idx

                input_graphs_raw = [sim[k].detach().cpu() for k in indices]

                # Construct input graph
                input_graph = build_velocity_graph_correction(input_graphs_raw, panic_at_positions=False).to(device)

                # Construct ModelInputs
                model_inputs = ModelInputs(
                    input_graphs_raw[-2].to(device) if history > 0 else input_graphs_raw[-1].to(device),
                    input_graphs_raw[-1].to(device),
                    sim[target_idx].to(device)
                )

                # Forward and Loss
                model_output = model(input_graph, is_training=True)
                acc_loss = huber_loss(model, model_output, model_inputs, is_training=True)

                # Backward
                loss_for_backward = acc_loss / accumulation_steps
                loss_for_backward.backward()

                # Optimization
                if (i + 1) % accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                total_acc_loss += acc_loss.item()
                train_samples += 1

            optimizer.step()
            optimizer.zero_grad()

        avg_train_loss = total_acc_loss / train_samples
        results["training_loss"].append(avg_train_loss)

        avg_val_pos_mse = 0.0

        # Validation
        if epoch % 5 == 0:
            real_ps, pred_ps, mses = [], [], []
            with torch.no_grad():
                model.eval()
                for val_sim in data['val'][:val_sims]:
                    input_graphs = [val_sim[i] for i in range(history+1)]
                    rollout = get_rollout(
                        input_graphs=input_graphs,
                        gnn_simulator=model,
                        gnn_history=history,
                        num_steps=num_rollout_steps,
                        barostat_config=barostat_config,
                        device="cuda"
                    )

                    pred_ps.append(calc_p_ratio_box_tensor(rollout)) 
                    real_ps.append(calc_p_ratio_box_tensor(val_sim[:len(rollout)]))
                    mses.append(torch.nn.functional.mse_loss(rollout[-1].x, val_sim[len(rollout)-1].x).item())

            r2 = r2_score(real_ps, pred_ps)
            results["r2"].append(r2)
            results["rollout_mse"].append(mses)
            
            # Calculate average MSE for logging
            if mses:
                avg_val_pos_mse = sum(mses) / len(mses)

            # Save model
            model.save_checkpoint(os.path.join(save_dir, f"checkpoint_epoch_{epoch}.pt"))

        lr_scheduler.step()
        t_stop = time.perf_counter()
        
        # Logging replaced the print statement
        logger.info(
            f"Epoch {epoch + 1:>3} | "
            f"Train Loss: {avg_train_loss:.3e} | "
            f"Val Pos MSE: {avg_val_pos_mse:.3e} | "
            f"r2 : {r2:.3f} "
            f"Time: {t_stop - t_start:.2f} s"
        )
        
    return results


def train_model_once_mst(data: dict[str, List], model: VelocityModel, epochs: int, barostat_config: Dict, save_dir: str, device: str = "cuda") -> Dict:
    
    freeze_norm_epoch = 5
    train_sims = 200
    val_sims = 200
    train_limit = 10
    accumulation_steps = 10
    learning_rate = 1e-3
    gamma = 0.995

    num_rollout_steps = 50
    history = 3

    # Limit training data
    poisson_threshold = 0.1
    training_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= poisson_threshold][:train_sims]
    logger.info(f"Using {len(training_data)} simulations with Poisson's ratio >= {poisson_threshold} for training.")
    logger.info(f"Using {len(data['val'][:val_sims])} simulations for validation.")

    params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.Adam(params, lr=learning_rate, weight_decay=0.0)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)

    results = {
        "training_loss": [], 
        "r2": [], 
        "rollout_mse": [] 
    }

    model.train()
    optimizer.zero_grad()

    r2 = 0.0

    for epoch in range(epochs):
        t_start = time.perf_counter()

        if epoch < 10:
            rollout_steps = 1
        elif epoch >= 10 and epoch < 20:
            rollout_steps = 2
        elif epoch >= 20 and epoch < 30:
            rollout_steps = 3
        elif epoch >=30:
            rollout_steps = 5
        else:
            rollout_steps = 5 # Catch-all for later epochs

        # Trackers
        total_acc_loss = 0
        train_samples = 0

        # Freeze normalizers
        if epoch == freeze_norm_epoch:
            model.node_normalizer.frozen = True
            model.edge_normalizer.frozen = True
            model.output_normalizer.frozen = True

        model.train()
        for sim in training_data:
            # Get equilibrium bond lengths
            r0 = sim[0].edge_attr[:, -2]
            
            starting_points = [i for i in range(train_limit)]
            if barostat_config["default_skip"] is not None:
                dump_period = barostat_config["default_skip"]
            else:
                sim_strain = (sim[1].box.x - sim[-1].box.x) / sim[0].box.x
                assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
                dump_period = int(assumed_rollout_length / len(sim)) + 1

            for i, start_idx in enumerate(starting_points):

                indices = [step + start_idx for step in range(history + 1)]
                current_window_graphs = [sim[k].detach().to(device) for k in indices]

                b0 = current_window_graphs[-2].box_tensor[0]
                b1 = current_window_graphs[-1].box_tensor[0]
                box_delta_x = b1 - b0

                if len(current_window_graphs) < 3:
                    current_box_vel_y = estimate_initial_box_vel_y(
                        current_window_graphs[-2],
                        current_window_graphs[-1],
                        dump_period * barostat_config["dt"],
                    )
                elif len(current_window_graphs) >= 3:
                    current_box_vel_y = estimate_initial_box_vel_y_accurate(
                        current_window_graphs[-3],
                        current_window_graphs[-2],
                        current_window_graphs[-1],
                        dump_period * barostat_config["dt"],
                    )
                else:
                    raise Exception(f"Window size is too small : {len(current_window_graphs)}")


                rollout_loss = 0
                for step in range(rollout_steps):
                    target_idx = history + 1 + start_idx + step
                    target_graph = sim[target_idx].to(device)

                    input_graph = build_velocity_graph_correction(current_window_graphs).to(device)

                    model_inputs = ModelInputs(
                        current_window_graphs[-2],
                        current_window_graphs[-1],
                        target_graph,
                    )

                    model_output = model(input_graph, is_training=True)
                    pred_graph_next = model.update(model_inputs, model_output)

                    step_loss = huber_loss(model, model_output, model_inputs, is_training=True)
                    rollout_loss += step_loss

                    dt = barostat_config["dt"]  # lammps dt
                    W_y = barostat_config["C_coupling"] * pred_graph_next.num_nodes * ((dump_period * dt) ** 2)
                    damping = barostat_config["damping"] * pred_graph_next.num_nodes * (dump_period * dt)

                    new_lx = pred_graph_next.box_tensor[0] + box_delta_x
                    new_ly, new_vel_y = update_box_y_thermodynamic(
                        positions=pred_graph_next.pos,
                        edge_index=model_inputs.cur_graph.edge_index,
                        edge_attr=model_inputs.cur_graph.edge_attr,
                        current_box=model_inputs.cur_graph.box_tensor,
                        r0=r0.to(pred_graph_next.pos.device),
                        box_vel_y=current_box_vel_y,  # Use ESTIMATED velocity
                        W_y=W_y,
                        damping=damping,
                        stride_dt=dump_period * dt,
                        target_pressure=barostat_config["target_pressure"],
                        temperature=barostat_config["temperature"],
                    )

                    new_box_tensor = torch.stack([new_lx, new_ly])
                    current_box_vel_y = new_vel_y

                    # Add new box and update edge_attr
                    pred_graph_next.box_tensor = new_box_tensor
                    pred_graph_next.edge_attr = get_correct_edge_attr(
                        pred_graph_next,
                        recompute_stiff=False,
                        panic_at_nontensor_box=True,
                    )
                    pred_graph_next.forces = compute_per_particle_forces(pred_graph_next, r0=r0.to(pred_graph_next.pos.device))

                    pred_graph_next_detached = pred_graph_next.detach()

                    # Update window: shift left, append new prediction
                    current_window_graphs.pop(0)
                    current_window_graphs.append(pred_graph_next_detached)

                # Divide by rollout_steps for average step loss, then by accumulation_steps
                final_loss = (rollout_loss / rollout_steps) / accumulation_steps
                final_loss.backward()

                
                if (i + 1) % accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                total_acc_loss += (final_loss.item() * accumulation_steps) # Re-multiply to get true magnitude for logging
                train_samples += 1

            # Catch remaining gradients for the sim if train_limit isn't divisible by accumulation_steps
            optimizer.step()
            optimizer.zero_grad()

        avg_train_loss = total_acc_loss / train_samples
        results["training_loss"].append(avg_train_loss)

        avg_val_pos_mse = 0.0

        # Validation
        if epoch % 5 == 0:
            real_ps, pred_ps, mses = [], [], []
            with torch.no_grad():
                model.eval()
                for val_sim in data['val'][:val_sims]:
                    input_graphs = [val_sim[i] for i in range(history+1)]
                    rollout = get_rollout(
                        input_graphs=input_graphs,
                        gnn_simulator=model,
                        gnn_history=history,
                        num_steps=num_rollout_steps,
                        barostat_config=barostat_config,
                        device="cuda"
                    )

                    pred_ps.append(calc_p_ratio_box_tensor(rollout)) 
                    real_ps.append(calc_p_ratio_box_tensor(val_sim[:len(rollout)]))
                    mses.append(torch.nn.functional.mse_loss(rollout[-1].x, val_sim[len(rollout)-1].x).item())

            r2 = r2_score(real_ps, pred_ps)
            results["r2"].append(r2)
            results["rollout_mse"].append(mses)
            
            # Calculate average MSE for logging
            if mses:
                avg_val_pos_mse = sum(mses) / len(mses)

            # Save model
            model.save_checkpoint(os.path.join(save_dir, f"checkpoint_epoch_{epoch}.pt"))
            model.train()

        lr_scheduler.step()
        t_stop = time.perf_counter()
        
        # Logging replaced the print statement
        logger.info(
            f"Epoch {epoch + 1:>3} | "
            f"Train Loss: {avg_train_loss:.3e} | "
            f"Val Pos MSE: {avg_val_pos_mse:.3e} | "
            f"r2 : {r2:.3f} "
            f"Time: {t_stop - t_start:.2f} s"
        )
        
    return results


In [ ]:
# Constant barostat params
barostat_config = barostat_parameters.node_optimizated

# Constant split
dataset_type = DatasetType.NodeOptimized
poisson_buckets = [
    {"max": 0.1, "count": 500},             # P < 0.1
    {"min": 0.1, "max": 0.2, "count": 500}, # 0.1 <= P < 0.2
    {"min": 0.2, "count": 300}              # P >= 0.2
]

main_directory = os.path.join("./cross_val", f"{dataset_type}", "MST")
os.makedirs(main_directory, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(main_directory, "training_crossval.log")),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# 1. Fetch paths along with their structural stratification labels
all_files, bucket_labels = get_sampled_files_with_labels(
    registry_path="./data/data_registry.csv",
    target_data_type=dataset_type,
    poisson_buckets=poisson_buckets,
    seed=42
)

# 2. Pre-load ALL data into memory exactly ONCE
logging.info(f"Pre-loading {len(all_files)} simulations into memory...")
max_sim_len = 100
master_dataset = []

for file in tqdm(all_files, desc="Loading & Preparing Data"):
    sim = torch.load(file, weights_only=False)[:max_sim_len]
    prepared_sim = prepare_traj(sim, calc_angles=False)
    master_dataset.append(prepared_sim)

logger.info("Data loaded successfully.")

k_folds = 5
seed = 42
# 3. Setup Stratified K-Fold Cross Validation
# This ensures fold splits closely match the target bucket ratios
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)

fold_results = []

# Pass bucket_labels into skf.split to enforce distribution balancing
for fold, (train_idx, val_idx) in enumerate(skf.split(master_dataset, bucket_labels)):
    logger.info(f"\n{'='*40}\nStarting Fold {fold + 1}/{k_folds} (Stratified)\n{'='*40}")
    
    data = {
        'train': [master_dataset[i] for i in train_idx],
        'val': [master_dataset[i] for i in val_idx]
    }

    logging.info(f"Loaded {len(data['train'])} training and {len(data['val'])} validation simulations.")

    # Optional check: verify that the proportion of buckets in data['val'] remains uniform
    val_labels_in_fold = [bucket_labels[i] for i in val_idx]
    logger.info(f"Fold {fold + 1} - Validation sample counts per bucket: "
                f"{ {b_id: val_labels_in_fold.count(b_id) for b_id in set(bucket_labels)} }")   

    # Init a new simulator model
    epochs = 100
    mp_layers = 2
    mlp = 3
    hidden_size = 128
    history = 3
    device = "cuda"

    init_graph = build_velocity_graph_correction(
        input_graphs=[data['train'][0][i].cpu().detach() for i in range(history + 1)],
        total_velocity=False,
        panic_at_positions=False
    ).to(device)

    gnn_simulator = VelocityModel(init_graph, hidden_size, mp_layers, mlp)
    gnn_simulator.to(device)

    fold_save_dir = os.path.join(main_directory, f"fold_{fold + 1}")
    os.makedirs(fold_save_dir, exist_ok=True)

    # Run training for this fold
    # results = train_model_once(data, gnn_simulator, epochs, barostat_config, fold_save_dir)
    results = train_model_once_mst(data, gnn_simulator, epochs, barostat_config, fold_save_dir, device=device)
    fold_results.append(results)
    gnn_simulator.save_checkpoint(os.path.join(fold_save_dir, f"model_full_range_h{history}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))
    torch.save(results, os.path.join(fold_save_dir, "results.pkl"))

    # Force immediate garbage collection to prevent GPU VRAM fragmentation
    del gnn_simulator
    torch.cuda.empty_cache()
    gc.collect()

torch.save(fold_results, main_directory)
